# JS08 - Tugas Lab 1: Hierarchical Clustering

Dataset: `CC_Customer_Dataset.csv` (660 data nasabah kartu kredit, 7 kolom, tanpa *missing value*).

Pada tugas ini kita membangun model klasterisasi **Agglomerative Clustering** menggunakan **seluruh variabel yang dapat digunakan**, kemudian memvisualisasikan bentuk *cluster* berdasarkan dua fitur `Avg_Credit_Limit` dan `Total_Credit_Cards`, serta mengevaluasi model dengan *silhouette score* dan *Davies-Bouldin Index* (DBI).

## Langkah 0 - Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import scipy.cluster.hierarchy as sch
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

## Langkah 1 - Load dan Inspeksi Data

In [ ]:
df = pd.read_csv('../../asset/CC_Customer_Dataset.csv')
df.head()

In [ ]:
df.info()

print('\nJumlah missing value per kolom:')
print(df.isnull().sum())

print('\nStatistik deskriptif:')
df.describe().round(2)

## Langkah 2 - Pemilihan Fitur dan Standardisasi

Variabel yang dapat digunakan adalah seluruh fitur numerik yang bermakna, yaitu `Avg_Credit_Limit`, `Total_Credit_Cards`, `Total_visits_bank`, `Total_visits_online`, dan `Total_calls_made`. Kolom `Sl_No` dan `Customer Key` tidak digunakan karena hanya berupa identitas baris/nasabah.

Karena skala fitur sangat berbeda (`Avg_Credit_Limit` bernilai ribuan hingga ratusan ribu, sedangkan fitur lain bernilai 0 - 15), data perlu distandarisasi terlebih dahulu dengan `StandardScaler`. Tanpa standardisasi, perhitungan jarak Euclidean pada Agglomerative Clustering akan didominasi oleh fitur `Avg_Credit_Limit`.

In [ ]:
fitur = ['Avg_Credit_Limit', 'Total_Credit_Cards',
         'Total_visits_bank', 'Total_visits_online', 'Total_calls_made']

X = df[fitur]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Shape data asli :', X.shape)
print('Shape data scaled:', X_scaled.shape)

print('\nStatistik fitur setelah standardisasi:')
pd.DataFrame(X_scaled, columns=fitur).describe().round(2)

## Langkah 3 - Visualisasi Dendrogram

Dendrogram dibangun dengan metode *linkage* **ward** untuk melihat struktur hierarki antardata. Tinggi garis vertikal menunjukkan jarak saat dua *cluster* digabung. Jumlah *cluster* dapat ditentukan dengan memotong dendrogram pada suatu *threshold* tertentu (garis horizontal).

In [ ]:
plt.figure(figsize=(12, 6))
dendrogram = sch.dendrogram(sch.linkage(X_scaled, method='ward'))
plt.title('Dendrogram (Ward Linkage)')
plt.xlabel('Nasabah')
plt.ylabel('Jarak Euclidean')
plt.show()

## Langkah 4 - Penentuan Jumlah Cluster Terbaik

Untuk memilih jumlah *cluster* secara objektif, kita menghitung *silhouette score* dan DBI untuk `k = 2` sampai `k = 10`. Kriteria terbaik adalah *silhouette score* tertinggi (mendekati +1) dan DBI terendah (mendekati 0).

In [ ]:
hasil = []

for k in range(2, 11):
    model = AgglomerativeClustering(n_clusters=k, metric='euclidean', linkage='ward')
    label = model.fit_predict(X_scaled)
    hasil.append({
        'k': k,
        'Silhouette': silhouette_score(X_scaled, label),
        'DBI': davies_bouldin_score(X_scaled, label)
    })

tabel_eval = pd.DataFrame(hasil).round(4)
tabel_eval

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))

ax1.plot(tabel_eval['k'], tabel_eval['Silhouette'], marker='o', color='tab:blue', label='Silhouette Score')
ax1.set_xlabel('Jumlah Cluster (k)')
ax1.set_ylabel('Silhouette Score', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.plot(tabel_eval['k'], tabel_eval['DBI'], marker='s', color='tab:red', label='DBI')
ax2.set_ylabel('Davies-Bouldin Index', color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title('Silhouette Score dan DBI untuk Berbagai Nilai k')
fig.tight_layout()
plt.show()

In [ ]:
k_best_silhouette = int(tabel_eval.loc[tabel_eval['Silhouette'].idxmax(), 'k'])
k_best_dbi = int(tabel_eval.loc[tabel_eval['DBI'].idxmin(), 'k'])

print(f'k dengan silhouette score tertinggi : {k_best_silhouette}')
print(f'k dengan DBI terendah              : {k_best_dbi}')

k_best = k_best_silhouette
print(f'\nk terbaik yang dipilih             : {k_best}')

### Analisis Penentuan Jumlah Cluster

| k | Silhouette Score | DBI |
|---|------------------|-----|
| 2 | 0.4177 | 0.9326 |
| 3 | **0.5148** | **0.6810** |
| 4 | 0.3481 | 1.1493 |
| 5 | 0.2569 | 1.3851 |
| 6 | 0.2268 | 1.4109 |
| 7 | 0.2163 | 1.4394 |
| 8 | 0.2187 | 1.4489 |
| 9 | 0.1953 | 1.5466 |
| 10 | 0.1931 | 1.5537 |

**Nilai `k` terbaik adalah 3.** Baik dari sisi *silhouette score* maupun DBI, `k=3` memberikan hasil yang paling optimal:

- *Silhouette score* tertinggi dicapai pada `k=3` (0.5148), sedangkan pada `k>3` nilainya terus menurun hingga 0.1931 di `k=10`.
- DBI terendah juga dicapai pada `k=3` (0.6810), lalu memburuk (naik) seiring bertambahnya jumlah *cluster*.

Artinya menambah *cluster* melebihi 3 justru memecah kelompok yang seharusnya utuh (*over-splitting*), sehingga `k=3` dipilih. Hasil ini juga konsisten dengan temuan pada JS07 (K-Means) yang menunjukkan data nasabah membentuk tiga segmen berdasarkan besaran limit kredit.

## Langkah 5 - Model Agglomerative Clustering

In [ ]:
cl_agg = AgglomerativeClustering(n_clusters=k_best, metric='euclidean', linkage='ward')
y_agg = cl_agg.fit_predict(X_scaled)

print(f'Jumlah cluster: {k_best}')
print('Jumlah anggota tiap cluster:')
print(pd.Series(y_agg).value_counts().sort_index())

## Langkah 6 - Visualisasi Bentuk Cluster (Scatterplot)

Scatterplot berikut menggunakan dua variabel `Avg_Credit_Limit` dan `Total_Credit_Cards` (nilai asli) dan mewarnai titik berdasarkan label *cluster* hasil model Agglomerative Clustering yang dibangun dari seluruh variabel.

In [ ]:
plt.figure(figsize=(8, 5))

for c in np.unique(y_agg):
    mask = y_agg == c
    plt.scatter(X['Avg_Credit_Limit'][mask], X['Total_Credit_Cards'][mask],
                s=40, alpha=0.7, label=f'Cluster {c}')

plt.xlabel('Avg_Credit_Limit')
plt.ylabel('Total_Credit_Cards')
plt.title('Bentuk Cluster Berdasarkan Avg_Credit_Limit dan Total_Credit_Cards')
plt.legend()
plt.show()

### Analisis Visualisasi

Scatterplot menunjukkan tiga kelompok nasabah yang terpisah terutama pada sumbu `Avg_Credit_Limit`:

- **Cluster 1** (~225 nasabah): limit kredit rendah (±12.000), jumlah kartu sedikit.
- **Cluster 0** (~385 nasabah): limit kredit menengah (±34.000), jumlah kartu sedang.
- **Cluster 2** (~50 nasabah): limit kredit tinggi (±141.000), jumlah kartu paling banyak.

Pola ini memperlihatkan bahwa meskipun model dibangun dari lima variabel, kedua fitur `Avg_Credit_Limit` dan `Total_Credit_Cards` saja sudah cukup kuat menampilkan pemisahan antar segmen nasabah.

## Langkah 7 - Evaluasi Model (Silhouette Score dan DBI)

In [ ]:
print(f'Silhouette Score     : {silhouette_score(X_scaled, y_agg):.4f}')
print(f'Davies-Bouldin Index : {davies_bouldin_score(X_scaled, y_agg):.4f}')

In [ ]:
profil = df.copy()
profil['cluster'] = y_agg

print('Rata-rata fitur tiap cluster:')
profil.groupby('cluster')[fitur].mean().round(2)

### Analisis Evaluasi

| Metrik | Nilai |
|--------|-------|
| Silhouette Score | 0.5148 |
| Davies-Bouldin Index | 0.6810 |

- **Silhouette Score 0.5148** menunjukkan *cluster* cukup padat dan terpisah antar kelompok (nilainya berada di atas 0 dan mendekati batas sedang-baik).
- **DBI 0.6810** relatif rendah sehingga separasi antar *cluster* cukup baik (semakin kecil semakin baik).

Profil rata-rata tiap *cluster* memperkuat interpretasi berikut:

- **Cluster 0 (385 nasabah, limit menengah):** rata-rata limit 33.852 dengan 5,5 kartu; aktif mengunjungi bank (3,5 kunjungan) dan jarang layanan *online*.
- **Cluster 1 (225 nasabah, limit rendah):** rata-rata limit 12.151 dengan 2,4 kartu; cenderung menggunakan layanan *online* (3,6) dan telepon (6,9) dibanding datang ke bank.
- **Cluster 2 (50 nasabah, limit tinggi):** rata-rata limit 141.040 dengan 8,7 kartu; sangat aktif menggunakan layanan *online* (10,9) dan jarang menghubungi telepon.

Nilai evaluasi ini berbeda dengan K-Means/DBSCAN sebelumnya karena cluster dibentuk dari lima variabel yang distandarisasi, bukan hanya dua fitur. Secara keseluruhan model Agglomerative Clustering berhasil membagi nasabah menjadi tiga segmen yang bermakna.